
# Exp6 — High-Dimensional Target-Subset and Candidate-Cap Robustness

### Purpose
> Could the controlled Adaptive-vs.-Shared conclusions be an artifact of using at most 32 target channels and a 128-source screening pool on high-dimensional datasets?

This experiment keeps the **published controlled-predictive-utility protocol unchanged** and perturbs only the two computational screening choices that a reviewer could question.

### Primary robustness test — target subset

For **Electricity** and **PeMS04**:

- original protocol: 32 approximately evenly spaced target channels;
- robustness protocol: five predeclared random 32-target subsets;
- source candidate pool: unchanged training-only Top-128 absolute current-value correlation;
- all standard horizons are evaluated;
- Shared and Adaptive selection use the same \(K_{\rm sel}=10\).

### Secondary robustness test — candidate-pool cap

Using the original evenly spaced 32 targets, repeat the experiment with source-candidate caps:

\[
64,\quad 128,\quad 192.
\]

This is preferable to randomly replacing source candidates, because the original pool is **not an arbitrary random subset**: it is a horizon-invariant training-only correlation screen. The robustness question is therefore whether the numerical cap matters.

### Fixed seeds

Random target-subset seeds are fixed **before observing results**:

`2026, 2027, 2028, 2029, 2030`.

### Main outputs

1. Adaptive-vs.-Shared endpoint-MSE gain for every target/horizon.
2. Condition-level mean/median gain and target win fraction.
3. Across-target-subset-seed mean ± std, min/max, and positive-seed fraction.
4. Candidate-cap sensitivity on the same fixed targets.
5. A reproduction check against the manuscript's original cap-128 values.

The experiment is intentionally limited to two representative high-dimensional datasets:
Electricity (general forecasting) and PeMS04 (traffic).


### v2 correction — reproduction constants

The first version used manuscript values from a different controlled comparison as the screen reference.  
The **actual released controlled-utility notebook** produces the following original deterministic-subset means:

- Electricity: `-2.0016, -0.3444, -0.4590, -0.4778` for H=`96,192,336,720`
- PeMS04: `-1.1511, -1.5881, +12.4078` for H=`12,24,48`

Your screen run reproduced these values exactly. The experiment implementation itself did **not** need correction; only the reference constants used by the automatic PASS/WARNING check were wrong.


In [1]:

from pathlib import Path
import gc, math, time, warnings, random
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path("/data/code/2026_08")
OUTPUT_DIR = PROJECT_ROOT / "results_highdim_subset_cap_robustness"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEQ_LEN = 96
TOPK = 10
MAX_TARGET_CHANNELS = 32
MAX_TRAIN_ORIGINS = 5000
MAX_TEST_ORIGINS = 3000

TEACHER_FIT_FRACTION = 0.70
RIDGE_LAMBDA = 1e-3
SOURCE_RIDGE_LAMBDA = 1e-2

TARGET_SEEDS = [2026, 2027, 2028, 2029, 2030]
CANDIDATE_CAPS = [64, 128, 192]

# Screen validates one exact-original scenario.
# Full performs all predeclared random-target and cap-robustness runs.
RUN_MODE = "full"  # "screen" or "full"

DATASET_REGISTRY = {
    "Electricity": {
        "family": "General",
        "horizons": [96, 192, 336, 720],
        "paths": [
            "/data/dataset/electricity/electricity.csv",
            "/data/dataset/electricity.csv",
            "/data/dataset/ECL.csv",
        ],
    },
    "PeMS04": {
        "family": "PeMS",
        "horizons": [12, 24, 48],
        "paths": [
            "/data/dataset/PeMS04.npz",
            "/data/dataset/PEMS04/PEMS04.npz",
            "/data/dataset/PEMS04.npz",
            "/data/dataset/pems/PeMS04.npz",
        ],
    },
}

# Manuscript values from the released controlled-selection analysis.
# Used only as a reproduction check, never for tuning.
# Exact values reproduced by the released Rethinking controlled-utility notebook.
# These are condition means over the original deterministic target subset.
EXPECTED_ORIGINAL_MEAN_GAIN = {
    ("Electricity", 96): -2.0016,
    ("Electricity", 192): -0.3444,
    ("Electricity", 336): -0.4590,
    ("Electricity", 720): -0.4778,
    ("PeMS04", 12): -1.1511,
    ("PeMS04", 24): -1.5881,
    ("PeMS04", 48): 12.4078,
}

print("OUTPUT_DIR:", OUTPUT_DIR)
print("RUN_MODE:", RUN_MODE)
print("TARGET_SEEDS:", TARGET_SEEDS)
print("CANDIDATE_CAPS:", CANDIDATE_CAPS)


OUTPUT_DIR: /data/code/2026_08/results_highdim_subset_cap_robustness
RUN_MODE: full
TARGET_SEEDS: [2026, 2027, 2028, 2029, 2030]
CANDIDATE_CAPS: [64, 128, 192]


## 1. Protocol-matched data and Ridge helpers

In [2]:

def resolve_path(candidates):
    for p in candidates:
        p = Path(p)
        if p.exists():
            return p
    return None

rows = []
for name, spec in DATASET_REGISTRY.items():
    p = resolve_path(spec["paths"])
    rows.append({
        "dataset": name,
        "path": None if p is None else str(p),
        "found": p is not None,
        "horizons": str(spec["horizons"]),
    })
path_df = pd.DataFrame(rows)
display(path_df)

if not path_df["found"].all():
    print("\nWARNING: edit DATASET_REGISTRY before running the experiment.")


,dataset,path,found,horizons
0,Electricity,/data/dataset/electricity/electricity.csv,True,"[96, 192, 336, 720]"
1,PeMS04,/data/dataset/PeMS04.npz,True,"[12, 24, 48]"


In [3]:
def load_series(
    dataset_name,
    path,
):
    path = Path(path)

    if path.suffix.lower() == ".npz":
        obj = np.load(
            path
        )

        if "data" in obj:
            x = obj[
                "data"
            ]

        else:
            key = list(
                obj.keys()
            )[0]

            x = obj[
                key
            ]

        # Common PeMS format: [T, C, F].
        if x.ndim == 3:
            x = x[
                :,
                :,
                0,
            ]

        elif x.ndim != 2:
            raise ValueError(
                f"{dataset_name}: unexpected NPZ shape {x.shape}"
            )

        x = x.astype(
            np.float32
        )

    elif path.suffix.lower() == ".csv":
        df = pd.read_csv(
            path
        )

        date_col = None

        for candidate in [
            "date",
            "datetime",
            "timestamp",
            "time",
        ]:
            if candidate in df.columns:
                date_col = candidate
                break

        if date_col is None:
            first = df.columns[
                0
            ]

            if not pd.api.types.is_numeric_dtype(
                df[
                    first
                ]
            ):
                date_col = first

        if date_col is not None:
            df = df.drop(
                columns=[
                    date_col
                ]
            )

        df = df.select_dtypes(
            include=[
                np.number
            ]
        )

        x = df.to_numpy(
            dtype=np.float32
        )

    else:
        try:
            x = np.loadtxt(
                path,
                delimiter=",",
                dtype=np.float32,
            )

        except Exception:
            x = np.loadtxt(
                path,
                dtype=np.float32,
            )

        if x.ndim == 1:
            x = x[
                :,
                None
            ]

    if x.ndim != 2:
        raise ValueError(
            f"{dataset_name}: expected [T,C], got {x.shape}"
        )

    if not np.isfinite(
        x
    ).all():
        # Simple interpolation-like fill is deliberately avoided.
        # Reject data so preprocessing remains explicit.
        raise ValueError(
            f"{dataset_name}: non-finite values detected"
        )

    return x


In [4]:
def split_boundaries(
    dataset_name,
    T,
):
    if dataset_name == "ETTh1":
        train_end = 12 * 30 * 24
        val_end = train_end + 4 * 30 * 24
        test_end = val_end + 4 * 30 * 24

        return (
            min(train_end, T),
            min(val_end, T),
            min(test_end, T),
        )

    if dataset_name == "ETTm1":
        unit = 30 * 24 * 4

        train_end = 12 * unit
        val_end = train_end + 4 * unit
        test_end = val_end + 4 * unit

        return (
            min(train_end, T),
            min(val_end, T),
            min(test_end, T),
        )

    # Same broad protocol used in the previous cross-domain experiments.
    return (
        int(
            0.70 * T
        ),
        int(
            0.80 * T
        ),
        T,
    )


def standardize_train_only(
    x,
    train_end,
):
    mean = x[
        :train_end
    ].mean(
        axis=0,
        keepdims=True,
    )

    std = x[
        :train_end
    ].std(
        axis=0,
        keepdims=True,
    )

    std = np.maximum(
        std,
        1e-6,
    )

    return (
        (
            x - mean
        )
        /
        std
    ).astype(
        np.float32
    )


In [5]:
def evenly_subsample(
    values,
    max_n,
):
    values = np.asarray(
        values,
        dtype=np.int64,
    )

    if len(
        values
    ) <= max_n:
        return values

    idx = np.linspace(
        0,
        len(
            values
        ) - 1,
        max_n,
        dtype=np.int64,
    )

    return values[
        idx
    ]


def make_origins(
    split_start,
    split_end,
    max_h,
    seq_len,
    max_n,
):
    first = max(
        seq_len,
        int(
            split_start
        ),
    )

    last = (
        int(
            split_end
        )
        -
        int(
            max_h
        )
    )

    if last < first:
        raise RuntimeError(
            f"No valid origins: start={first}, end={last}"
        )

    origins = np.arange(
        first,
        last + 1,
        dtype=np.int64,
    )

    return evenly_subsample(
        origins,
        max_n,
    )


def summary_features(
    x,
    origins,
):
    """
    Returns [N,C,4]:
      last
      mean3
      mean12
      delta12
    """
    origins = np.asarray(
        origins,
        dtype=np.int64,
    )

    last = x[
        origins - 1
    ]

    mean3 = np.stack(
        [
            x[
                t-3:
                t
            ].mean(
                axis=0
            )
            for t in origins
        ],
        axis=0,
    )

    mean12 = np.stack(
        [
            x[
                t-12:
                t
            ].mean(
                axis=0
            )
            for t in origins
        ],
        axis=0,
    )

    delta12 = (
        x[
            origins - 1
        ]
        -
        x[
            origins - 12
        ]
    )

    return np.stack(
        [
            last,
            mean3,
            mean12,
            delta12,
        ],
        axis=-1,
    ).astype(
        np.float32
    )


In [6]:
def add_bias_column(
    X,
):
    return np.concatenate(
        [
            np.ones(
                (
                    len(
                        X
                    ),
                    1,
                ),
                dtype=np.float32,
            ),
            X.astype(
                np.float32
            ),
        ],
        axis=1,
    )


def ridge_fit(
    X,
    y,
    lam=RIDGE_LAMBDA,
):
    Xb = add_bias_column(
        X
    )

    p = Xb.shape[
        1
    ]

    reg = np.eye(
        p,
        dtype=np.float64,
    )

    # Do not regularize intercept.
    reg[
        0,
        0
    ] = 0.0

    A = (
        Xb.T
        @
        Xb
    ).astype(
        np.float64
    )

    b = (
        Xb.T
        @
        y
    ).astype(
        np.float64
    )

    beta = np.linalg.solve(
        A
        +
        lam
        *
        reg,
        b,
    )

    return beta


def ridge_predict(
    X,
    beta,
):
    return (
        add_bias_column(
            X
        )
        @
        beta
    ).astype(
        np.float32
    )


def fit_and_mse(
    X_train,
    y_train,
    X_test,
    y_test,
    lam=RIDGE_LAMBDA,
):
    beta = ridge_fit(
        X_train,
        y_train,
        lam=lam,
    )

    pred = ridge_predict(
        X_test,
        beta,
    )

    return float(
        np.mean(
            (
                y_test
                -
                pred
            )
            **
            2
        )
    )


In [7]:
def _safe_topk_indices(
    scores,
    k,
):
    scores = np.asarray(
        scores,
        dtype=np.float64,
    )

    k = min(
        int(
            k
        ),
        len(
            scores
        ),
    )

    if k <= 0:
        return np.array(
            [],
            dtype=np.int64,
        )

    idx = np.argpartition(
        scores,
        -k,
    )[
        -k:
    ]

    return idx[
        np.argsort(
            scores[
                idx
            ]
        )[
            ::-1
        ]
    ]


def _jaccard(
    a,
    b,
):
    a = set(
        np.asarray(
            a
        ).tolist()
    )

    b = set(
        np.asarray(
            b
        ).tolist()
    )

    if len(
        a | b
    ) == 0:
        return 1.0

    return (
        len(
            a & b
        )
        /
        len(
            a | b
        )
    )


def predictive_utility_for_target(
    X_train_all,
    y_train,
    target_idx,
    source_ids,
):
    """
    Training-only teacher.

    X_train_all: [N,C,4]
    y_train:      [N]

    Steps:
      1) chronologically split official train origins into fit/score
      2) fit target-only baseline on target features
      3) residualize every source's 4 features against target features
      4) fit marginal residual correction for all sources in batched 4x4 solves
      5) score utility on held-out teacher-score segment
      6) compute source-set reliability by scoring the same fitted corrections
         on first/second halves of teacher-score segment
    """
    N = X_train_all.shape[
        0
    ]

    split = int(
        TEACHER_FIT_FRACTION
        *
        N
    )

    split = min(
        max(
            split,
            32,
        ),
        N - 16,
    )

    fit_idx = np.arange(
        0,
        split,
        dtype=np.int64,
    )

    score_idx = np.arange(
        split,
        N,
        dtype=np.int64,
    )

    Xt = X_train_all[
        :,
        target_idx,
        :
    ].astype(
        np.float64
    )

    U = X_train_all[
        :,
        source_ids,
        :
    ].astype(
        np.float64
    )

    y = y_train.astype(
        np.float64
    )

    # --------------------------------------------------------
    # Target-only baseline.
    # --------------------------------------------------------
    beta_t = ridge_fit(
        Xt[
            fit_idx
        ],
        y[
            fit_idx
        ],
        lam=RIDGE_LAMBDA,
    )

    pred_fit = ridge_predict(
        Xt[
            fit_idx
        ],
        beta_t,
    ).astype(
        np.float64
    )

    pred_score = ridge_predict(
        Xt[
            score_idx
        ],
        beta_t,
    ).astype(
        np.float64
    )

    r_fit = (
        y[
            fit_idx
        ]
        -
        pred_fit
    )

    r_score = (
        y[
            score_idx
        ]
        -
        pred_score
    )

    base_mse = float(
        np.mean(
            r_score
            *
            r_score
        )
    )

    # --------------------------------------------------------
    # Residualize all source features against target history.
    #
    # Multi-output regression:
    #   source_features ~ target_features
    #
    # This prevents utility from simply reflecting information
    # already present in the target channel.
    # --------------------------------------------------------
    Xt_fit_b = add_bias_column(
        Xt[
            fit_idx
        ]
    ).astype(
        np.float64
    )

    Xt_score_b = add_bias_column(
        Xt[
            score_idx
        ]
    ).astype(
        np.float64
    )

    U_fit = U[
        fit_idx
    ]

    U_score = U[
        score_idx
    ]

    S = U_fit.shape[
        1
    ]

    Fdim = U_fit.shape[
        2
    ]

    U_fit_flat = U_fit.reshape(
        len(
            fit_idx
        ),
        S
        *
        Fdim,
    )

    A = (
        Xt_fit_b.T
        @
        Xt_fit_b
    )

    reg = np.eye(
        A.shape[
            0
        ],
        dtype=np.float64,
    )

    reg[
        0,
        0
    ] = 0.0

    coef_u = np.linalg.solve(
        A
        +
        RIDGE_LAMBDA
        *
        reg,
        Xt_fit_b.T
        @
        U_fit_flat,
    )

    U_fit_res = (
        U_fit_flat
        -
        Xt_fit_b
        @
        coef_u
    ).reshape(
        len(
            fit_idx
        ),
        S,
        Fdim,
    )

    U_score_flat = U_score.reshape(
        len(
            score_idx
        ),
        S
        *
        Fdim,
    )

    U_score_res = (
        U_score_flat
        -
        Xt_score_b
        @
        coef_u
    ).reshape(
        len(
            score_idx
        ),
        S,
        Fdim,
    )

    # --------------------------------------------------------
    # Batched marginal residual correction for every source.
    # --------------------------------------------------------
    gram = np.einsum(
        "nsf,nsg->sfg",
        U_fit_res,
        U_fit_res,
    )

    cross = np.einsum(
        "nsf,n->sf",
        U_fit_res,
        r_fit,
    )

    eye = np.eye(
        Fdim,
        dtype=np.float64,
    )[
        None,
        :,
        :
    ]

    beta_s = np.linalg.solve(
        gram
        +
        SOURCE_RIDGE_LAMBDA
        *
        eye,
        cross[
            :,
            :,
            None
        ],
    )[
        :,
        :,
        0
    ]

    pred_res_score = np.einsum(
        "nsf,sf->ns",
        U_score_res,
        beta_s,
    )

    err = (
        r_score[
            :,
            None
        ]
        -
        pred_res_score
    )

    mse_source = np.mean(
        err
        *
        err,
        axis=0,
    )

    utility = (
        100.0
        *
        (
            base_mse
            -
            mse_source
        )
        /
        max(
            base_mse,
            1e-12,
        )
    )

    # --------------------------------------------------------
    # Reliability: score same fitted marginal models on the two
    # chronological halves of the teacher-score segment.
    # --------------------------------------------------------
    n_score = len(
        score_idx
    )

    mid = max(
        1,
        n_score // 2,
    )

    def utility_slice(
        start,
        end,
    ):
        rr = r_score[
            start:
            end
        ]

        pp = pred_res_score[
            start:
            end
        ]

        if len(
            rr
        ) == 0:
            return utility.copy()

        base = np.mean(
            rr
            *
            rr
        )

        mse = np.mean(
            (
                rr[
                    :,
                    None
                ]
                -
                pp
            )
            **
            2,
            axis=0,
        )

        return (
            100.0
            *
            (
                base
                -
                mse
            )
            /
            max(
                base,
                1e-12,
            )
        )

    utility_first = utility_slice(
        0,
        mid,
    )

    utility_second = utility_slice(
        mid,
        n_score,
    )

    return {
        "source_ids": np.asarray(
            source_ids,
            dtype=np.int64,
        ),
        "utility": utility.astype(
            np.float32
        ),
        "utility_first": utility_first.astype(
            np.float32
        ),
        "utility_second": utility_second.astype(
            np.float32
        ),
        "teacher_base_mse": base_mse,
    }


In [8]:
def feature_matrix_for_sources(
    X_all,
    target_idx,
    source_ids,
):
    target = X_all[
        :,
        target_idx,
        :
    ]

    if len(
        source_ids
    ) == 0:
        return target

    source = X_all[
        :,
        source_ids,
        :
    ].reshape(
        len(
            X_all
        ),
        -1,
    )

    return np.concatenate(
        [
            target,
            source,
        ],
        axis=1,
    )


def selected_forecast_mse(
    X_train_all,
    y_train,
    X_test_all,
    y_test,
    target_idx,
    source_ids,
):
    Xtr = feature_matrix_for_sources(
        X_train_all,
        target_idx,
        source_ids,
    )

    Xte = feature_matrix_for_sources(
        X_test_all,
        target_idx,
        source_ids,
    )

    return fit_and_mse(
        Xtr,
        y_train,
        Xte,
        y_test,
        lam=RIDGE_LAMBDA,
    )


## 2. Robust target subsets and parameterized candidate cap

In [9]:

def choose_targets_even(C):
    if C <= MAX_TARGET_CHANNELS:
        return np.arange(C, dtype=np.int64)
    return np.unique(
        np.linspace(0, C - 1, MAX_TARGET_CHANNELS, dtype=np.int64)
    )

def choose_targets_random(C, seed):
    if C <= MAX_TARGET_CHANNELS:
        return np.arange(C, dtype=np.int64)
    rng = np.random.default_rng(int(seed))
    return np.sort(
        rng.choice(C, size=MAX_TARGET_CHANNELS, replace=False)
    ).astype(np.int64)

def source_candidate_map_with_cap(train_features, targets, cap):
    # Exact original screening principle:
    # training-only absolute current-value correlation, horizon invariant.
    # Only the numerical cap is parameterized.
    current = train_features[:, :, 0].astype(np.float64)
    current = current - current.mean(axis=0, keepdims=True)
    denom = np.sqrt(np.sum(current * current, axis=0))
    denom = np.maximum(denom, 1e-12)
    normed = current / denom[None, :]

    C = current.shape[1]
    pool_size = min(int(cap), max(1, C - 1))
    result = {}

    for i0 in targets:
        i = int(i0)
        corr = np.abs(normed[:, i] @ normed)
        corr[i] = -np.inf
        idx = np.argpartition(corr, -pool_size)[-pool_size:]
        idx = idx[np.argsort(corr[idx])[::-1]]
        result[i] = idx.astype(np.int64)

    return result

def safe_spearman(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    ok = np.isfinite(a) & np.isfinite(b)
    if ok.sum() < 3 or np.std(a[ok]) < 1e-12 or np.std(b[ok]) < 1e-12:
        return np.nan
    r = spearmanr(a[ok], b[ok])
    if hasattr(r, "statistic"):
        return float(r.statistic)
    if hasattr(r, "correlation"):
        return float(r.correlation)
    return float(r[0])


## 3. Prepare each dataset once

In [10]:

def prepare_dataset(dataset_name):
    spec = DATASET_REGISTRY[dataset_name]
    path = resolve_path(spec["paths"])
    if path is None:
        raise FileNotFoundError(dataset_name)

    raw = load_series(dataset_name, path)
    T, C = raw.shape
    horizons = list(spec["horizons"])
    max_h = max(horizons)

    train_end, val_end, test_end = split_boundaries(dataset_name, T)
    x = standardize_train_only(raw, train_end)

    train_origins = make_origins(
        split_start=SEQ_LEN,
        split_end=train_end,
        max_h=max_h,
        seq_len=SEQ_LEN,
        max_n=MAX_TRAIN_ORIGINS,
    )
    test_origins = make_origins(
        split_start=val_end,
        split_end=test_end,
        max_h=max_h,
        seq_len=SEQ_LEN,
        max_n=MAX_TEST_ORIGINS,
    )

    X_train = summary_features(x, train_origins)
    X_test = summary_features(x, test_origins)

    print(
        f"{dataset_name}: T={T}, C={C}, "
        f"train_origins={len(train_origins)}, test_origins={len(test_origins)}"
    )

    return {
        "dataset": dataset_name,
        "family": spec["family"],
        "horizons": horizons,
        "x": x,
        "T": T,
        "C": C,
        "train_origins": train_origins,
        "test_origins": test_origins,
        "X_train": X_train,
        "X_test": X_test,
    }

prepared = {}
for d in DATASET_REGISTRY:
    prepared[d] = prepare_dataset(d)


Electricity: T=26304, C=321, train_origins=5000, test_origins=3000
PeMS04: T=16992, C=307, train_origins=5000, test_origins=3000


## 4. One robustness run

In [11]:

def analyze_subset_run(prep, target_strategy, target_seed, candidate_cap):
    t0 = time.time()
    dataset_name = prep["dataset"]
    horizons = prep["horizons"]
    x = prep["x"]
    C = prep["C"]
    train_origins = prep["train_origins"]
    test_origins = prep["test_origins"]
    X_train = prep["X_train"]
    X_test = prep["X_test"]

    if target_strategy == "even":
        targets = choose_targets_even(C)
        target_seed_value = -1
    elif target_strategy == "random":
        targets = choose_targets_random(C, target_seed)
        target_seed_value = int(target_seed)
    else:
        raise ValueError(target_strategy)

    candidate_map = source_candidate_map_with_cap(
        X_train, targets, candidate_cap
    )

    teacher_cache = {}

    for ti, target_idx0 in enumerate(targets):
        target_idx = int(target_idx0)
        source_ids = candidate_map[target_idx]
        utility_by_h = {}

        for h in horizons:
            y_train = x[train_origins + int(h) - 1, target_idx]
            utility_by_h[int(h)] = predictive_utility_for_target(
                X_train, y_train, target_idx, source_ids
            )

        teacher_cache[target_idx] = utility_by_h

        if (ti + 1) % 8 == 0 or (ti + 1) == len(targets):
            print(
                f"  {dataset_name} | {target_strategy} seed={target_seed_value} "
                f"| cap={candidate_cap} | teacher {ti+1}/{len(targets)}"
            )

    rows = []

    for target_idx0 in targets:
        target_idx = int(target_idx0)
        utility_by_h = teacher_cache[target_idx]
        source_ids = utility_by_h[horizons[0]]["source_ids"]

        utility_stack = np.stack(
            [utility_by_h[h]["utility"] for h in horizons],
            axis=0,
        )
        shared_utility = utility_stack.mean(axis=0)
        shared_local = _safe_topk_indices(shared_utility, TOPK)
        shared_sources = source_ids[shared_local]

        for h in horizons:
            teacher = utility_by_h[h]
            utility = teacher["utility"]
            adaptive_local = _safe_topk_indices(utility, TOPK)
            adaptive_sources = source_ids[adaptive_local]

            y_train = x[train_origins + int(h) - 1, target_idx]
            y_test = x[test_origins + int(h) - 1, target_idx]

            target_only_mse = selected_forecast_mse(
                X_train, y_train, X_test, y_test, target_idx, []
            )
            shared_mse = selected_forecast_mse(
                X_train, y_train, X_test, y_test, target_idx, shared_sources
            )
            adaptive_mse = selected_forecast_mse(
                X_train, y_train, X_test, y_test, target_idx, adaptive_sources
            )

            adaptive_gain = 100.0 * (
                shared_mse - adaptive_mse
            ) / max(shared_mse, 1e-12)

            rows.append({
                "dataset": dataset_name,
                "family": prep["family"],
                "target_strategy": target_strategy,
                "target_seed": target_seed_value,
                "candidate_cap": int(candidate_cap),
                "target_channel": target_idx,
                "horizon": int(h),
                "target_only_test_mse": float(target_only_mse),
                "shared_test_mse": float(shared_mse),
                "adaptive_test_mse": float(adaptive_mse),
                "adaptive_gain_vs_shared_%": float(adaptive_gain),
                "predictive_utility_topk_mean_%": float(
                    np.mean(utility[adaptive_local])
                ),
                "source_reliability": float(
                    _jaccard(
                        source_ids[_safe_topk_indices(teacher["utility_first"], TOPK)],
                        source_ids[_safe_topk_indices(teacher["utility_second"], TOPK)],
                    )
                ),
                "shared_sources": ",".join(map(str, shared_sources.tolist())),
                "adaptive_sources": ",".join(map(str, adaptive_sources.tolist())),
            })

    df = pd.DataFrame(rows)
    print(
        f"DONE {dataset_name} | {target_strategy} seed={target_seed_value} "
        f"| cap={candidate_cap} | rows={len(df)} | "
        f"{time.time()-t0:.1f}s"
    )
    return df


## 5. Run screen or full robustness suite

In [12]:

run_specs = []

if RUN_MODE == "screen":
    # Exact original protocol on one dataset. Use this to verify reproduction first.
    run_specs = [
        ("Electricity", "even", -1, 128),
    ]
else:
    for dataset in ["Electricity", "PeMS04"]:
        # Exact-original reference.
        run_specs.append((dataset, "even", -1, 128))

        # Primary robustness: five random target subsets at fixed original cap 128.
        for seed in TARGET_SEEDS:
            run_specs.append((dataset, "random", seed, 128))

        # Secondary cap robustness on the exact original target subset.
        for cap in [64, 192]:
            run_specs.append((dataset, "even", -1, cap))

print("Number of runs:", len(run_specs))
display(pd.DataFrame(
    run_specs,
    columns=["dataset", "target_strategy", "target_seed", "candidate_cap"]
))

frames = []
failures = []

for dataset, target_strategy, target_seed, cap in run_specs:
    tag = f"{dataset}_{target_strategy}_seed{target_seed}_cap{cap}"
    out_csv = OUTPUT_DIR / f"{tag}_rows.csv"

    if out_csv.exists():
        print("Reusing:", out_csv)
        frames.append(pd.read_csv(out_csv))
        continue

    try:
        df = analyze_subset_run(
            prepared[dataset],
            target_strategy=target_strategy,
            target_seed=target_seed,
            candidate_cap=cap,
        )
        df.to_csv(out_csv, index=False)
        frames.append(df)
    except Exception as e:
        print("FAILED:", tag, repr(e))
        failures.append({"run": tag, "error": repr(e)})

    gc.collect()

if not frames:
    raise RuntimeError("No robustness run completed.")

all_rows = pd.concat(frames, ignore_index=True)
all_rows.to_csv(
    OUTPUT_DIR / f"{RUN_MODE}_highdim_robustness_all_rows.csv",
    index=False,
)

failure_df = pd.DataFrame(failures)
failure_df.to_csv(
    OUTPUT_DIR / f"{RUN_MODE}_failures.csv",
    index=False,
)

print("\nTotal rows:", len(all_rows))
print("Failures:", len(failures))
if len(failure_df):
    display(failure_df)


Number of runs: 16


,dataset,target_strategy,target_seed,candidate_cap
0,Electricity,even,-1,128
1,Electricity,random,2026,128
2,Electricity,random,2027,128
3,Electricity,random,2028,128
4,Electricity,random,2029,128
5,Electricity,random,2030,128
6,Electricity,even,-1,64
7,Electricity,even,-1,192
8,PeMS04,even,-1,128
9,PeMS04,random,2026,128


Reusing: /data/code/2026_08/results_highdim_subset_cap_robustness/Electricity_even_seed-1_cap128_rows.csv
  Electricity | random seed=2026 | cap=128 | teacher 8/32
  Electricity | random seed=2026 | cap=128 | teacher 16/32
  Electricity | random seed=2026 | cap=128 | teacher 24/32
  Electricity | random seed=2026 | cap=128 | teacher 32/32
DONE Electricity | random seed=2026 | cap=128 | rows=128 | 15.1s
  Electricity | random seed=2027 | cap=128 | teacher 8/32
  Electricity | random seed=2027 | cap=128 | teacher 16/32
  Electricity | random seed=2027 | cap=128 | teacher 24/32
  Electricity | random seed=2027 | cap=128 | teacher 32/32
DONE Electricity | random seed=2027 | cap=128 | rows=128 | 15.9s
  Electricity | random seed=2028 | cap=128 | teacher 8/32
  Electricity | random seed=2028 | cap=128 | teacher 16/32
  Electricity | random seed=2028 | cap=128 | teacher 24/32
  Electricity | random seed=2028 | cap=128 | teacher 32/32
DONE Electricity | random seed=2028 | cap=128 | rows=128 | 

## 6. Condition-level results

In [13]:

condition_summary = (
    all_rows
    .groupby(
        [
            "dataset",
            "target_strategy",
            "target_seed",
            "candidate_cap",
            "horizon",
        ],
        as_index=False,
    )
    .agg(
        n_targets=("target_channel", "nunique"),
        mean_gain_pct=("adaptive_gain_vs_shared_%", "mean"),
        median_gain_pct=("adaptive_gain_vs_shared_%", "median"),
        target_win_fraction=("adaptive_gain_vs_shared_%", lambda s: float(np.mean(np.asarray(s) > 0))),
        mean_topk_utility_pct=("predictive_utility_topk_mean_%", "mean"),
        mean_source_reliability=("source_reliability", "mean"),
    )
)

condition_summary.to_csv(
    OUTPUT_DIR / f"{RUN_MODE}_condition_summary.csv",
    index=False,
)

display(condition_summary.round(4))


,dataset,target_strategy,target_seed,candidate_cap,horizon,n_targets,mean_gain_pct,median_gain_pct,target_win_fraction,mean_topk_utility_pct,mean_source_reliability
0,Electricity,even,-1,64,96,32,-0.1538,0.0202,0.5312,8.8936,0.1812
1,Electricity,even,-1,64,192,32,0.3565,0.6864,0.5938,6.9143,0.1659
2,Electricity,even,-1,64,336,32,0.8011,1.3255,0.6250,6.0144,0.1204
3,Electricity,even,-1,64,720,32,-0.7850,0.9461,0.5312,9.0050,0.0974
4,Electricity,even,-1,128,96,32,-2.0016,-0.7384,0.4688,9.3572,0.1235
5,Electricity,even,-1,128,192,32,-0.3444,-0.4335,0.4375,7.2979,0.1113
6,Electricity,even,-1,128,336,32,-0.4590,-0.8329,0.4375,6.5082,0.0941
7,Electricity,even,-1,128,720,32,-0.4778,-1.4388,0.4688,9.8251,0.0572
8,Electricity,even,-1,192,96,32,-0.2049,0.1505,0.5625,9.6178,0.1006
9,Electricity,even,-1,192,192,32,0.2340,0.2187,0.5000,7.5156,0.1040


## 7. Reproduce the published original cap-128 conditions

In [14]:

original = condition_summary[
    (condition_summary["target_strategy"] == "even")
    & (condition_summary["candidate_cap"] == 128)
].copy()

original["expected_manuscript_gain_pct"] = [
    EXPECTED_ORIGINAL_MEAN_GAIN.get((d, int(h)), np.nan)
    for d, h in zip(original["dataset"], original["horizon"])
]
original["abs_difference_pctpt"] = np.abs(
    original["mean_gain_pct"] - original["expected_manuscript_gain_pct"]
)

display(original.round(4))
original.to_csv(
    OUTPUT_DIR / f"{RUN_MODE}_original_reproduction_check.csv",
    index=False,
)

if len(original):
    max_diff = float(original["abs_difference_pctpt"].max())
    print("Maximum absolute difference from manuscript:", round(max_diff, 6), "percentage points")
    if max_diff > 0.10:
        print("WARNING: reproduction differs by >0.10 percentage points.")
        print("Inspect paths/protocol before interpreting robustness results.")
    else:
        print("PASS: original cap-128 reproduction is within 0.10 percentage points.")


,dataset,target_strategy,target_seed,candidate_cap,horizon,n_targets,mean_gain_pct,median_gain_pct,target_win_fraction,mean_topk_utility_pct,mean_source_reliability,expected_manuscript_gain_pct,abs_difference_pctpt
4,Electricity,even,-1,128,96,32,-2.0016,-0.7384,0.4688,9.3572,0.1235,-2.0016,0.0
5,Electricity,even,-1,128,192,32,-0.3444,-0.4335,0.4375,7.2979,0.1113,-0.3444,0.0
6,Electricity,even,-1,128,336,32,-0.4590,-0.8329,0.4375,6.5082,0.0941,-0.4590,0.0
7,Electricity,even,-1,128,720,32,-0.4778,-1.4388,0.4688,9.8251,0.0572,-0.4778,0.0
35,PeMS04,even,-1,128,12,32,-1.1511,-0.1447,0.4688,29.8825,0.4848,-1.1511,0.0
36,PeMS04,even,-1,128,24,32,-1.5881,-0.6889,0.4375,32.7089,0.4789,-1.5881,0.0
37,PeMS04,even,-1,128,48,32,12.4078,8.7406,0.5938,31.8638,0.4855,12.4078,0.0


Maximum absolute difference from manuscript: 4.6e-05 percentage points
PASS: original cap-128 reproduction is within 0.10 percentage points.


## 8. Five-seed target-subset robustness

In [15]:

random128 = condition_summary[
    (condition_summary["target_strategy"] == "random")
    & (condition_summary["candidate_cap"] == 128)
].copy()

if len(random128):
    target_subset_robustness = (
        random128
        .groupby(["dataset", "horizon"], as_index=False)
        .agg(
            n_subset_seeds=("target_seed", "nunique"),
            mean_over_seeds_pct=("mean_gain_pct", "mean"),
            std_over_seeds_pct=("mean_gain_pct", "std"),
            min_seed_gain_pct=("mean_gain_pct", "min"),
            max_seed_gain_pct=("mean_gain_pct", "max"),
            median_over_seeds_pct=("mean_gain_pct", "median"),
            positive_seed_fraction=("mean_gain_pct", lambda s: float(np.mean(np.asarray(s) > 0))),
        )
    )
    target_subset_robustness.to_csv(
        OUTPUT_DIR / "full_target_subset_robustness_summary.csv",
        index=False,
    )
    display(target_subset_robustness.round(4))

    dataset_seed_average = (
        random128
        .groupby(["dataset", "target_seed"], as_index=False)
        .agg(
            horizon_avg_gain_pct=("mean_gain_pct", "mean"),
            positive_horizon_fraction=("mean_gain_pct", lambda s: float(np.mean(np.asarray(s) > 0))),
        )
    )
    dataset_seed_average.to_csv(
        OUTPUT_DIR / "full_target_subset_dataset_seed_summary.csv",
        index=False,
    )
    print("\nDataset-level horizon averages by random target subset:")
    display(dataset_seed_average.round(4))
else:
    print("Target-subset robustness requires RUN_MODE='full'.")


,dataset,horizon,n_subset_seeds,mean_over_seeds_pct,std_over_seeds_pct,min_seed_gain_pct,max_seed_gain_pct,median_over_seeds_pct,positive_seed_fraction
0,Electricity,96,5,-2.2206,3.9842,-8.9758,0.9913,-0.8787,0.4
1,Electricity,192,5,1.0515,1.2513,-0.6697,2.3087,1.6740,0.8
2,Electricity,336,5,-2.9815,2.5841,-5.5778,1.0747,-3.8676,0.2
3,Electricity,720,5,-2.2417,4.0690,-8.1242,2.4950,-2.8204,0.4
4,PeMS04,12,5,0.9254,2.2783,-1.5047,3.7005,-0.2408,0.4
5,PeMS04,24,5,-0.2161,1.5072,-2.5437,1.2679,0.2680,0.6
6,PeMS04,48,5,2.4737,5.9520,-7.1276,8.0929,3.1859,0.8



Dataset-level horizon averages by random target subset:


,dataset,target_seed,horizon_avg_gain_pct,positive_horizon_fraction
0,Electricity,2026,0.2763,0.7500
1,Electricity,2027,-2.7599,0.5000
2,Electricity,2028,-1.4056,0.5000
3,Electricity,2029,-1.1684,0.2500
4,Electricity,2030,-2.9326,0.2500
5,PeMS04,2026,3.2745,1.0000
6,PeMS04,2027,3.0400,0.6667
7,PeMS04,2028,-2.7480,0.0000
8,PeMS04,2029,2.5381,1.0000
9,PeMS04,2030,-0.7996,0.3333


## 9. Candidate-cap robustness on identical targets

In [16]:

even_caps = condition_summary[
    condition_summary["target_strategy"] == "even"
].copy()

if len(even_caps):
    cap_table = even_caps.pivot_table(
        index=["dataset", "horizon"],
        columns="candidate_cap",
        values="mean_gain_pct",
    ).reset_index()
    cap_table.to_csv(
        OUTPUT_DIR / f"{RUN_MODE}_candidate_cap_gain_table.csv",
        index=False,
    )
    display(cap_table.round(4))

    cap_pair_rows = []
    for dataset in even_caps["dataset"].unique():
        sub = all_rows[
            (all_rows["dataset"] == dataset)
            & (all_rows["target_strategy"] == "even")
        ]
        caps = sorted(sub["candidate_cap"].unique())
        for a_i in range(len(caps)):
            for b_i in range(a_i + 1, len(caps)):
                ca, cb = int(caps[a_i]), int(caps[b_i])
                A = sub[sub["candidate_cap"] == ca][
                    ["target_channel", "horizon", "adaptive_gain_vs_shared_%"]
                ].rename(columns={"adaptive_gain_vs_shared_%": "gain_a"})
                B = sub[sub["candidate_cap"] == cb][
                    ["target_channel", "horizon", "adaptive_gain_vs_shared_%"]
                ].rename(columns={"adaptive_gain_vs_shared_%": "gain_b"})
                m = A.merge(B, on=["target_channel", "horizon"], how="inner")
                cap_pair_rows.append({
                    "dataset": dataset,
                    "cap_a": ca,
                    "cap_b": cb,
                    "n_target_horizon": len(m),
                    "spearman_gain": safe_spearman(m["gain_a"], m["gain_b"]),
                    "sign_agreement": float(
                        np.mean(np.sign(m["gain_a"]) == np.sign(m["gain_b"]))
                    ),
                    "mean_abs_gain_difference_pctpt": float(
                        np.mean(np.abs(m["gain_a"] - m["gain_b"]))
                    ),
                })

    cap_pair_summary = pd.DataFrame(cap_pair_rows)
    cap_pair_summary.to_csv(
        OUTPUT_DIR / f"{RUN_MODE}_candidate_cap_pairwise_stability.csv",
        index=False,
    )
    print("\nTarget-horizon stability across candidate caps:")
    display(cap_pair_summary.round(4))


candidate_cap,dataset,horizon,64,128,192
0,Electricity,96,-0.1538,-2.0016,-0.2049
1,Electricity,192,0.3565,-0.3444,0.2340
2,Electricity,336,0.8011,-0.4590,0.9777
3,Electricity,720,-0.7850,-0.4778,-1.3532
4,PeMS04,12,-1.5283,-1.1511,-2.2233
5,PeMS04,24,-10.7143,-1.5881,-6.9457
6,PeMS04,48,-11.7751,12.4078,5.0064



Target-horizon stability across candidate caps:


,dataset,cap_a,cap_b,n_target_horizon,spearman_gain,sign_agreement,mean_abs_gain_difference_pctpt
0,Electricity,64,128,128,0.3318,0.6484,6.6508
1,Electricity,64,192,128,0.2122,0.5781,7.3626
2,Electricity,128,192,128,0.5245,0.7266,5.5150
3,PeMS04,64,128,96,0.5425,0.6771,23.1888
4,PeMS04,64,192,96,0.2328,0.5208,30.3747
5,PeMS04,128,192,96,0.2732,0.6146,18.2799


## 10. Fixed interpretation guide

In [17]:

print("=" * 96)
print("EXP6 INTERPRETATION — PREDECLARED ROBUSTNESS QUESTIONS")
print("=" * 96)

guide = """
Primary question:
  Does the Adaptive-vs.-Shared conclusion materially depend on the exact 32 target channels?

Evidence to report:
  - five-seed mean +/- std of condition mean gains,
  - min/max across subset seeds,
  - fraction of subset seeds with positive mean gain.

Secondary question:
  Does the 128-source correlation-screen cap determine the result?

Evidence to report:
  - mean gains at caps 64 / 128 / 192 on identical targets,
  - target-horizon gain-rank correlation and sign agreement across caps.

Interpretation rules:
  1) Stable direction and similar magnitude across target seeds:
       strong evidence against an arbitrary target-subset artifact.
  2) Similar conclusions across caps:
       evidence that cap=128 is not a fragile screening choice.
  3) Heterogeneous results:
       report the sensitivity explicitly and narrow the claim;
       do not select a favorable seed or cap after seeing results.
"""
print(guide)

print("Files to return:")
for p in sorted(OUTPUT_DIR.glob("*.csv")):
    print(" -", p)

if RUN_MODE == "screen":
    print("\nNEXT:")
    print("  If the reproduction check passes, set RUN_MODE='full' and rerun.")
    print("  Full mode is required regardless of whether the robustness outcome is favorable.")


EXP6 INTERPRETATION — PREDECLARED ROBUSTNESS QUESTIONS

Primary question:
  Does the Adaptive-vs.-Shared conclusion materially depend on the exact 32 target channels?

Evidence to report:
  - five-seed mean +/- std of condition mean gains,
  - min/max across subset seeds,
  - fraction of subset seeds with positive mean gain.

Secondary question:
  Does the 128-source correlation-screen cap determine the result?

Evidence to report:
  - mean gains at caps 64 / 128 / 192 on identical targets,
  - target-horizon gain-rank correlation and sign agreement across caps.

Interpretation rules:
  1) Stable direction and similar magnitude across target seeds:
       strong evidence against an arbitrary target-subset artifact.
  2) Similar conclusions across caps:
       evidence that cap=128 is not a fragile screening choice.
  3) Heterogeneous results:
       report the sensitivity explicitly and narrow the claim;
       do not select a favorable seed or cap after seeing results.

Files to retur